# Phase 7: Wide ResNet Migration & Squeezing (Quantization)
This notebook takes your existing `.pth` Wide ResNet model from your previous geospatial project, converts it to ONNX, mathematically simplifies the graph to avoid PyTorch exporter bugs, and squeezes its file size by 75% using INT8 Dynamic Quantization so it can run blazingly fast in the React Web App.

In [ ]:
!pip install torch torchvision onnx onnxruntime onnxscript onnxsim
from google.colab import drive
drive.mount('/content/drive')

## The Ultimate Export Pipeline
**User Action Required:** 
Update the `MODEL_PATH` variable below to point to exactly where your `.pth` file is in your Google Drive.

In [ ]:
import torch
import torchvision.models as models
import os
import onnx
import onnxsim
from onnxruntime.quantization import quantize_dynamic, QuantType

# >>> UPDATE THIS PATH TO YOUR .pth FILE IN GOOGLE DRIVE <<<
MODEL_PATH = '/content/drive/MyDrive/geospatial/best_indian_model.pth'

# 1. Load the model and clean the keys
checkpoint = torch.load(MODEL_PATH, map_location=torch.device('cpu'))
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

if 'fc.0.weight' in state_dict:
    state_dict['fc.weight'] = state_dict.pop('fc.0.weight')
    state_dict['fc.bias'] = state_dict.pop('fc.0.bias')

model = models.wide_resnet50_2(num_classes=6)
model.load_state_dict(state_dict)
model.eval()
print("✅ PyTorch Model Loaded!")

# 2. Export to standard ONNX (FP32)
dummy_input = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, 
    dummy_input, 
    "wideresnet_raw.onnx", 
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'], 
    output_names=['output']
)
print("✅ Raw ONNX Exported!")

# 3. Forcefully simplify the graph to fix the PyTorch exporter bug
model_onnx = onnx.load("wideresnet_raw.onnx")
model_simp, check = onnxsim.simplify(model_onnx)
if not check:
    print("Warning: Simplifier failed check, but continuing...")
onnx.save(model_simp, "wideresnet_simp.onnx")
print("✅ ONNX Graph Simplified!")

# 4. Squeeze it!
print("Squeezing model into INT8...")
quantize_dynamic("wideresnet_simp.onnx", "wideresnet_quantized.onnx", weight_type=QuantType.QUInt8)

original_size = os.path.getsize("wideresnet_raw.onnx") / (1024 * 1024)
squeezed_size = os.path.getsize("wideresnet_quantized.onnx") / (1024 * 1024)

print(f"\n📊 Original Size: {original_size:.2f} MB")
print(f"📊 Squeezed Size: {squeezed_size:.2f} MB")
print("✅ Quantization complete! Downloading to your computer...")

from google.colab import files
files.download('wideresnet_quantized.onnx')